# Session 11: Policies and Procedures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-11-policies-and-procedures.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-11_policy_audit.py`

In [1]:
"""Session 11 - Audit a switch/router configuration against a hardening policy.

Reads a saved configuration (session-11_router.cfg, a made-up lab device) and
checks it against the organisation's baseline. This is what configuration
compliance tools do every night.
Run:  python session-11_policy_audit.py
"""
import re

CHECKS = [  # (id, description, regex that MUST be present or must be absent, must_exist)
    ("H1", "Passwords stored with strong secret (not 'enable password')", r"^enable (algorithm-type \S+ )?secret", True),
    ("H2", "No plain 'enable password' line", r"^enable password", False),
    ("H3", "Password encryption service on", r"^service password-encryption", True),
    ("H4", "SSH version 2 only", r"^ip ssh version 2", True),
    ("H5", "Telnet not allowed on VTY lines", r"transport input .*telnet", False),
    ("H6", "Login banner present", r"^banner (motd|login)", True),
    ("H7", "HTTP server disabled", r"^no ip http server", True),
    ("H8", "SNMP default community 'public' not used", r"^snmp-server community public", False),
    ("H9", "Logging to a central syslog server", r"^logging host \S+", True),
    ("H10", "NTP configured", r"^ntp server \S+", True),
    ("H11", "Idle sessions time out (exec-timeout not 0 0)", r"exec-timeout 0 0", False),
    ("H12", "Unused interfaces shut down", None, True),
]


def unused_up(cfg):
    """Interfaces described as UNUSED that are not shut down."""
    bad = []
    for block in re.split(r"\n(?=interface )", cfg):
        if block.startswith("interface") and "UNUSED" in block and "shutdown" not in block:
            bad.append(block.splitlines()[0].split()[1])
    return bad


def audit(cfg):
    results = []
    for cid, desc, rx, must in CHECKS:
        if cid == "H12":
            bad = unused_up(cfg)
            results.append((cid, desc, not bad, ", ".join(bad)))
            continue
        found = re.search(rx, cfg, re.M) is not None
        results.append((cid, desc, found == must, ""))
    return results


if __name__ == "__main__":
    try:
        cfg = open("session-11_router.cfg").read()
    except FileNotFoundError:          # e.g. running on Colab: fetch the sample from the repo
        import urllib.request
        url = ("https://raw.githubusercontent.com/Hamnakhan123/Cybersecurity-and-Digital-Forensics/"
               "main/term-1/03-computer-networks/code/session-11_router.cfg")
        cfg = urllib.request.urlopen(url).read().decode()
    res = audit(cfg)
    for cid, desc, ok, extra in res:
        print(f"{'PASS' if ok else 'FAIL'}  {cid:<4} {desc}{'  -> ' + extra if extra else ''}")
    score = sum(r[2] for r in res)
    print(f"\nCompliance: {score}/{len(res)} = {score / len(res):.0%}")

FAIL  H1   Passwords stored with strong secret (not 'enable password')
FAIL  H2   No plain 'enable password' line
FAIL  H3   Password encryption service on
PASS  H4   SSH version 2 only
FAIL  H5   Telnet not allowed on VTY lines
PASS  H6   Login banner present
FAIL  H7   HTTP server disabled
FAIL  H8   SNMP default community 'public' not used
PASS  H9   Logging to a central syslog server
PASS  H10  NTP configured
FAIL  H11  Idle sessions time out (exec-timeout not 0 0)
FAIL  H12  Unused interfaces shut down  -> GigabitEthernet0/2

Compliance: 4/12 = 33%


## `session-11_password_policy.py`

In [2]:
"""Session 11 - Check passwords against an acceptable-use / password policy
and estimate their strength. Modern guidance (e.g. NIST SP 800-63B) favours
length and a block-list over forced complexity and frequent changes.
Run:  python session-11_password_policy.py
"""
import math
import string

BLOCKLIST = {"password", "123456", "qwerty", "admin", "welcome", "cisco", "letmein"}
POLICY = {"min_length": 12, "max_length": 64}


def entropy_bits(pw):
    pool = 0
    if any(c in string.ascii_lowercase for c in pw): pool += 26
    if any(c in string.ascii_uppercase for c in pw): pool += 26
    if any(c in string.digits for c in pw): pool += 10
    if any(c in string.punctuation or c == " " for c in pw): pool += 33
    return len(pw) * math.log2(pool) if pool else 0


def check(pw, username):
    problems = []
    if len(pw) < POLICY["min_length"]: problems.append(f"shorter than {POLICY['min_length']}")
    if len(pw) > POLICY["max_length"]: problems.append("too long")
    low = pw.lower()
    if any(b in low for b in BLOCKLIST): problems.append("contains a block-listed word")
    if username.lower() in low: problems.append("contains the username")
    if len(set(pw)) <= 3: problems.append("too repetitive")
    return problems


if __name__ == "__main__":
    tests = [("hamna", "Cisco123"), ("hamna", "P@ssw0rd2026!"), ("ali", "ali-network-2026"),
             ("sara", "aaaaaaaaaaaaaaa"), ("lab", "river tulip lantern seventy"), ("lab", "x7#Qp!2vR9$m")]
    for user, pw in tests:
        p = check(pw, user)
        verdict = "ACCEPT" if not p else "REJECT: " + "; ".join(p)
        print(f"{pw!r:<32} ~{entropy_bits(pw):>5.0f} bits  {verdict}")

'Cisco123'                       ~   48 bits  REJECT: shorter than 12; contains a block-listed word
'P@ssw0rd2026!'                  ~   85 bits  ACCEPT
'ali-network-2026'               ~   98 bits  REJECT: contains the username
'aaaaaaaaaaaaaaa'                ~   71 bits  REJECT: too repetitive
'river tulip lantern seventy'    ~  159 bits  ACCEPT
'x7#Qp!2vR9$m'                   ~   79 bits  ACCEPT
